# Mathematical Correctness Proof: `core_logistic_regression_lasso.py`

This notebook proves that the `fit` method of our custom L1-regularized logistic regression actually does what it claims to do: minimize an L1-penalized weighted logistic loss using proximal gradient descent.

The argument is a **loop invariant** proof. The idea is straightforward: we state a property that should hold at the start of every iteration of the training loop. If that property is true before the first iteration, and if running one iteration preserves it, then it holds forever.

This style of argument is called **induction**. If you have not seen induction before, the idea is this. Imagine a long row of dominoes. If you can show that (1) the first domino falls, and (2) any falling domino knocks over the next one, then all the dominoes will fall, no matter how long the row is. You did not have to check each domino individually. Mathematical induction works the same way: you show a property holds in the simplest case (the **base case**), and you show that if it holds for one case it must hold for the next (the **inductive step**). Together, those two facts give you the property for every case at once. In our setting, the "first domino" is "before any training runs", and "one domino knocks over the next" is "running one iteration of the loop preserves the property". Once both facts are nailed down, we get correctness for the entire training run.

----------

## What we are minimizing

Before introducing the math, here is the big picture. A machine learning model is a function with adjustable knobs (called **parameters** or **coefficients**). Training means finding the right setting of those knobs so the model's predictions match the labels in our data. We measure how badly the current settings are doing using a number called the **cost** (or **loss**, or **objective**). Lower cost means better settings. Training is the process of finding the knob settings that make the cost as small as possible.

Our cost function is

$$J(w, b) = L(w, b) + \alpha \|w\|_1$$

The function has two pieces that pull in opposite directions. The first piece, $L(w, b)$, is the data-fitting loss. It is small when the model's predictions match the labels and large when they do not. The second piece, $\alpha \|w\|_1 = \alpha \sum_j |w_j|$, is a penalty on the total absolute size of the coefficients.

The notation $\|w\|_1$ is called the **L1 norm** of the vector $w$. It just means: add up the absolute values of every entry of $w$. For example, if $w = (3, -2, 0, 1)$, then $\|w\|_1 = |3| + |{-2}| + |0| + |1| = 6$. The L1 norm is a way of measuring how "big" a vector is in total magnitude, treating positive and negative entries equally. Multiplying by $\alpha$ scales this penalty: bigger $\alpha$ means the penalty matters more.

Higher $\alpha$ therefore punishes large coefficients more harshly, which pushes weak features to exactly zero and leaves only the ones that really earn their keep. This is why the model is called "Lasso": the L1 penalty is so harsh on small coefficients that it "lassoes" them down to exactly zero. Unlike L2 (Ridge) regularization, which shrinks coefficients smoothly toward zero without ever quite reaching it, L1 produces **sparse** solutions: many coefficients become exactly zero, which makes the model easier to interpret because you can simply ignore the features whose weight is zero.

----------

## Variables and symbols

A quick orientation on notation before the symbol list. We use **matrices** (rectangular grids of numbers), **vectors** (single rows or columns of numbers), and **scalars** (single numbers). The notation $\mathbb{R}$ means "the set of real numbers" (all the numbers you encounter in school: positive, negative, fractions, decimals, but not imaginary numbers). When we write $X \in \mathbb{R}^{n \times p}$, the symbol $\in$ means "is in" or "is an element of", and $\mathbb{R}^{n \times p}$ means "the set of $n$-by-$p$ matrices of real numbers". Read together: "$X$ is a matrix of real numbers with $n$ rows and $p$ columns."

With that out of the way, here is each symbol used in this proof, with a sentence or two about what it actually represents in our project.

**$X \in \mathbb{R}^{n \times p}$, the feature matrix.** A big table of numbers with $n$ rows and $p$ columns. Each row corresponds to one training comment, and each column corresponds to one engineered feature (profanity count, VADER sentiment score, identity-mention flag, and so on). In our setup, $n$ is about 127,000 training comments and $p$ is about 32 dense features (the ones built in `build_features.py`). The notation $X^\top$ means the **transpose** of $X$: rows and columns swapped, so $X^\top \in \mathbb{R}^{p \times n}$. Concretely, if the entry in row $i$, column $j$ of $X$ is the value of feature $j$ for comment $i$, then the entry in row $j$, column $i$ of $X^\top$ is the same value. The transpose shows up naturally when you take derivatives of matrix expressions.

**$y \in \{0, 1\}^n$, the label vector.** A vector of length $n$ where each entry is either 0 or 1. $y_i = 1$ if comment $i$ is toxic, $y_i = 0$ if it is not. In our data the toxic class is rare (about 9% of training rows), which is why sample weighting becomes important. The notation $\{0, 1\}^n$ means "a sequence of $n$ values, each of which is either 0 or 1".

**$w \in \mathbb{R}^p$, the coefficient vector.** This is the actual thing the model is learning. There is one weight $w_j$ per feature, so $w$ has $p$ entries. Each weight tells you how much its feature contributes to the model's decision: if $w_j$ is large and positive, increasing feature $j$ makes the model more confident the comment is toxic; if $w_j$ is large and negative, increasing feature $j$ makes the model lean toward non-toxic; if $w_j$ is zero, the model ignores feature $j$ entirely. After training, this vector is stored as `self.coef_`.

**$b \in \mathbb{R}$, the intercept.** A single real number (a scalar), not a vector. It acts as the model's baseline. If every feature were exactly zero, the prediction would be $\sigma(b)$, where $\sigma$ is the sigmoid function explained below. After training, it is stored as `self.intercept_`.

**$\hat{p} \in [0, 1]^n$, the vector of predicted probabilities.** A vector of length $n$ where $\hat{p}_i$ is the model's estimated probability that comment $i$ is toxic. The formula is $\hat{p}_i = \sigma(x_i^\top w + b)$. Let us unpack this:

- $x_i$ is the $i$-th row of $X$, viewed as a vector. So $x_i$ is "the features of comment $i$" packaged into a single object with $p$ entries.
- $x_i^\top w$ is the **dot product** of $x_i$ and $w$. The dot product of two vectors of the same length is computed by multiplying their entries pairwise and adding everything up: if $x_i = (x_{i,1}, x_{i,2}, \ldots, x_{i,p})$ and $w = (w_1, w_2, \ldots, w_p)$, then $x_i^\top w = x_{i,1} w_1 + x_{i,2} w_2 + \cdots + x_{i,p} w_p$. It produces a single number: a weighted sum of features, where each feature is multiplied by its own weight.
- Adding $b$ shifts the result up or down by the intercept.
- $\sigma$ is the **sigmoid function**, defined by $\sigma(z) = \frac{1}{1 + e^{-z}}$. Its graph is an S-shape that smoothly squashes any input (no matter how positive or negative) into the interval $[0, 1]$. Very negative inputs come out close to 0, very positive inputs come out close to 1, and an input of 0 comes out exactly at 0.5. Because the output is always between 0 and 1, we can read it as a probability.

So $\hat{p}_i$ is computed by: take the features of comment $i$, multiply each by its weight, add them all up, add the intercept, then squash through the sigmoid to get a number in $[0, 1]$ that we interpret as the probability of toxicity.

**$\mathbf{sw} \in \mathbb{R}_{\geq 0}^n$, the sample-weight vector.** A vector of length $n$ with non-negative entries (the subscript $\geq 0$ indicates non-negativity). If the user does not pass `sample_weight` to `fit`, every entry defaults to 1, meaning every comment counts equally. With class imbalance like ours (9% toxic), we typically pass weights that scale up the rare class so it contributes more to the gradient. Without this, a lazy model could just predict "non-toxic" for every comment and still get 91% accuracy.

**$\alpha \geq 0$, the L1 regularization strength.** A non-negative scalar the user picks. Larger $\alpha$ means a harsher penalty on coefficient magnitudes, which causes more coefficients to be driven to exactly zero. If $\alpha = 0$, there is no penalty and we recover ordinary logistic regression. If $\alpha$ is enormous, every coefficient gets zeroed and the model becomes useless. The sweet spot is in the middle and we find it via `GridSearchCV`.

**$\eta > 0$, the learning rate (step size).** A positive scalar that controls how big a step the algorithm takes in each iteration. Too small means slow convergence; too large means the algorithm overshoots the minimum and bounces around. In the code this is `self.learning_rate`.

**$L(w, b)$, the weighted logistic loss.** A function that takes the current coefficients and intercept and returns a single number measuring how badly the model's predictions disagree with the labels (weighted by $\mathbf{sw}$). Lower is better. The logistic loss is **convex** in $w$ and $b$. Convex means, intuitively, that the graph of the function looks like a bowl: there is one bottom point, and from anywhere on the bowl, going "downhill" eventually takes you to that bottom. Convex functions are great for optimization because they have **no local minima other than the global one**: you cannot get stuck in a side-valley because there are no side-valleys.

**$J(w, b) = L(w, b) + \alpha \|w\|_1$, the total cost.** What the algorithm actually minimizes. The first piece pushes for accuracy, the second piece pushes for sparsity, and $\alpha$ balances the two. Because both $L$ and $\|w\|_1$ are convex, their sum $J$ is also convex.

**$S_\lambda(z)$, the soft-thresholding operator.** Takes a vector $z$ and a non-negative threshold $\lambda$, returns a new vector with each entry shrunk toward zero. Used in the sub-step (b) below. Full definition there.

**Superscript notation $(t)$.** When we write $w^{(t)}$, we mean "the value of $w$ at the start of iteration $t$." So $w^{(0)}$ is the initial value (all zeros, before any training), $w^{(1)}$ is the value after one update step, $w^{(2)}$ is the value after two steps, and so on. The superscript is a counter for the iteration index, not an exponent. We are NOT raising $w$ to a power.

----------

## What is proximal gradient descent, and why do we need it?

### Plain gradient descent (the starting point)

Before introducing the "proximal" version, here is how plain gradient descent works in general. This part requires a quick detour into what a **gradient** is, since the whole method is built on it.

A **function** in our context is a recipe that takes some inputs and returns one number. For example, $f(w)$ takes a vector $w$ of inputs and returns a single number. The cost $J(w, b)$ from above is a function: give it some coefficients and an intercept, and it gives you back the total cost.

To **minimize** a function means: find the inputs that make the output as small as possible. For a one-dimensional function (one input, one output), this is like finding the lowest point on a curve.

The **gradient** of $f$ at a point $w$, written $\nabla f(w)$, is a vector. It has one entry for each input variable. Roughly speaking, the gradient tells you, for each input, "if you wiggle this input upward, here is how much the output goes up". A positive entry means wiggling that input up makes the function go up; a negative entry means wiggling that input up makes the function go down; a zero entry means the function is flat in that direction at this point. Bigger numbers (in absolute value) mean steeper changes.

For a one-dimensional function, the gradient is just the slope of the graph at that point (in calculus this is called the **derivative**). For a multi-dimensional function, the gradient is a vector of partial derivatives, one for each input. You can picture it as an arrow pointing in the direction of steepest **ascent** (the direction in which the function rises fastest).

With that in hand, here is the gradient descent update rule:

$$w \gets w - \eta \, \nabla f(w)$$

Let us read this expression piece by piece:

- $w$ on the right-hand side is the current value of the coefficient vector.
- $\nabla f(w)$ is the gradient of $f$ evaluated at the current $w$. It is a vector pointing in the direction of steepest **ascent** (where $f$ rises fastest).
- $\eta$ is the learning rate, a small positive scalar. It controls how big a step we take.
- The minus sign flips the direction: we step opposite the gradient, which is the direction of steepest **descent**. Going downhill is what we want when minimizing.
- $\eta \, \nabla f(w)$ is therefore the displacement we apply: a small step in the downhill direction.
- The symbol $\gets$ means **assignment**: replace the old value on the left with the new value computed on the right.

In words: "Look at the slope where you are standing. Go a little way in the downhill direction. Repeat." After enough iterations, $w$ settles near a point where the gradient is zero, which is a minimum (for a convex function, the global minimum, because convex functions only have one valley).

### Why this recipe breaks for our problem

Plain gradient descent assumes the function we are minimizing is **smooth** at every point. "Smooth" is shorthand for "the gradient exists everywhere": at every possible input, you can write down a single, unambiguous slope (in one dimension) or gradient vector (in higher dimensions). Without smoothness, the recipe breaks at the very first step of each iteration, which is "compute the gradient". If the gradient does not exist at our current location, the algorithm has no defined step to take.

The logistic loss $L(w, b)$ on its own is smooth. Take any input, and you can compute a unique gradient cleanly. So if we were minimizing just $L$, plain gradient descent would work fine.

But our full cost is $J(w, b) = L(w, b) + \alpha \|w\|_1$. The L1 term $\|w\|_1 = \sum_j |w_j|$ is a sum of absolute-value functions, one per coefficient. The absolute value $|w_j|$ is **not** smooth at $w_j = 0$. To see why, picture the graph of $|w_j|$ as a function of a single variable $w_j$:

- For $w_j > 0$, the graph is the line $w_j$. The slope is $+1$ everywhere on this side.
- For $w_j < 0$, the graph is the line $-w_j$. The slope is $-1$ everywhere on this side.
- At exactly $w_j = 0$, the two lines meet at a sharp V-corner.

At that corner there is no single slope. If you ask "what is the derivative of $|w_j|$ at $w_j = 0$?", the answer from the left is $-1$, the answer from the right is $+1$, and the two disagree. The derivative is undefined, which means the gradient of the full $J$ is also undefined whenever any single coefficient sits at zero.

You might wonder: why is this such a problem? Why not pretend the slope at the corner is something convenient (like 0) and carry on? Because the corner is exactly where L1 wants to drive coefficients. The whole point of L1 regularization is that the optimal solution often has many coefficients equal to zero. That is what makes the model sparse. So the corner is not some weird edge case we can dodge; it is where most of the action happens.

Common naive workarounds tend to misbehave near zero in concrete ways:

- **Pretend the derivative is 0 at the corner.** Then the algorithm sees no penalty force at $w_j = 0$ and ignores the L1 term there. Coefficients that should be pulled to zero stay slightly non-zero, and the sparsity property disappears.
- **Pretend the derivative is some constant like $+1$.** Then the algorithm always pushes the coefficient in one direction, which is wrong half the time (the coefficient could have been on either side of zero).
- **Smooth the absolute value by replacing $|w_j|$ with something rounded like $\sqrt{w_j^2 + \varepsilon}$.** Now the gradient exists everywhere and plain gradient descent works, but the smoothed function no longer has a sharp corner, so the algorithm shrinks coefficients toward zero but never reaches exactly zero. Again, sparsity is lost.

What we actually want is a method that handles the corner exactly, not one that smooths it away or guesses. That method is proximal gradient descent.

### The proximal fix

Proximal gradient descent splits one iteration of the algorithm into two halves, so we end up applying two updates instead of one. The trick is that each half deals with only the "easy" part of the problem: Half 1 handles the smooth logistic loss using plain gradient descent, and Half 2 handles the non-smooth L1 penalty using a special operator. Neither half ever has to compute a gradient at the broken corner.

**Half 1: take a gradient step on the smooth part only.**

The first half computes the gradient of $L$ alone (the L1 penalty is ignored at this stage) and takes a step in the opposite direction:

$$z^{(t+1)} = w^{(t)} - \eta \, \nabla_w L(w^{(t)}, b^{(t)})$$

Let us read the right-hand side carefully:

- $w^{(t)}$ is the current coefficient vector at the start of iteration $t$. Recall that the superscript $(t)$ is an iteration counter, not an exponent.
- $\nabla_w L(w^{(t)}, b^{(t)})$ is the gradient of the loss $L$ evaluated at the current $(w^{(t)}, b^{(t)})$. The subscript $w$ on $\nabla$ means "take the gradient with respect to $w$ only" (we treat $b$ as a fixed constant for this step). The result is a vector of length $p$, one entry per coefficient.
- Multiplying by $\eta$ scales that gradient down by the learning rate. Bigger $\eta$ means a bigger step.
- The minus sign flips the direction so we move opposite of the gradient (downhill in $L$).
- Subtracting from $w^{(t)}$ produces a new vector, which we call $z^{(t+1)}$.

What is $z^{(t+1)}$? It is an **intermediate value**, not the final coefficient vector yet. It is what the coefficient vector would become if we cared only about minimizing $L$ and the L1 penalty did not exist. Think of it as "the step we would take if the problem were easy" or "Half 1's answer".

**Half 2: apply the proximal operator.**

The second half corrects the intermediate $z^{(t+1)}$ to account for the L1 penalty:

$$w^{(t+1)} = S_{\eta \alpha}(z^{(t+1)})$$

The pieces here:

- $z^{(t+1)}$ is the intermediate value from Half 1.
- $S_{\eta \alpha}$ is the **soft-thresholding operator** with threshold $\lambda = \eta \alpha$. We define it precisely in sub-step (b) below. For now it is enough to know it takes a vector and returns a new vector by shrinking each entry toward zero by up to $\eta \alpha$, snapping entries to zero whenever they are small.
- Applying $S_{\eta \alpha}$ to $z^{(t+1)}$ produces $w^{(t+1)}$, the new coefficient vector that becomes "current" at the start of iteration $t + 1$.

So one full iteration takes $w^{(t)}$ to $w^{(t+1)}$ in two stages: first a gradient step in the smooth loss $L$ to get the intermediate $z^{(t+1)}$, then a soft-threshold to incorporate the L1 penalty into $w^{(t+1)}$. The intermediate $z^{(t+1)}$ is "the move you would make if there were no L1 penalty", and the soft-threshold is "the correction that bakes the L1 penalty in".

There is a theorem from **convex optimization** (a field of mathematics that studies how to minimize convex functions) that guarantees: performing Half 1 followed by Half 2 is mathematically equivalent to performing one correct gradient step on the full L1-penalized objective $J$. So the trick really does solve the corner problem, not just sidestep it. We will not prove this theorem here, but it is the foundational result behind every proximal-gradient method, and it is in any standard convex-optimization textbook.

**The intercept update.**

The intercept $b$ is updated separately, with no soft-thresholding:

$$b^{(t+1)} = b^{(t)} - \eta \, \nabla_b L(w^{(t)}, b^{(t)})$$

The reason: $b$ does not appear inside the L1 penalty $\alpha \|w\|_1$ (the penalty applies only to the coefficient vector $w$). So the non-smooth corner has nothing to do with $b$ at all. Plain gradient descent on $b$ works perfectly well, and we have no reason to add a proximal step here. The update is just "current $b$ minus a learning-rate-scaled gradient", exactly like one-dimensional gradient descent.

----------

## The loop invariant we want to prove

Now that we know what one iteration looks like, here is the property we want to prove about the entire training loop.

> **Invariant (I):** At the start of iteration $t$ of the training loop, `self.coef_` contains exactly the value $w^{(t)}$ that you would get from $t$ correct proximal gradient steps on the coefficients, and `self.intercept_` contains exactly the value $b^{(t)}$ that you would get from $t$ correct gradient steps on the intercept. Formally, the entire history $w^{(1)}, w^{(2)}, \ldots, w^{(t)}$ and $b^{(1)}, b^{(2)}, \ldots, b^{(t)}$ must obey the proximal-gradient recurrence at every step: for every $k = 1, 2, \ldots, t$,
>
> $$z^{(k)} = w^{(k-1)} - \eta \, \nabla_w L(w^{(k-1)}, b^{(k-1)})$$
>
> $$w^{(k)} = S_{\eta \alpha}(z^{(k)})$$
>
> $$b^{(k)} = b^{(k-1)} - \eta \, \nabla_b L(w^{(k-1)}, b^{(k-1)})$$

Let us decode this. The phrase "for every $k = 1, 2, \ldots, t$" means we are listing one condition per past iteration. If $t = 0$, the list is empty and the invariant says nothing (no steps have happened yet, so there is nothing to check). If $t = 1$, the list has one condition: the update from $w^{(0)}, b^{(0)}$ to $w^{(1)}, b^{(1)}$. If $t = 5$, the list has five conditions, one per step that has happened.

Each of the three lines describes one piece of a single step:

- The first line, $z^{(k)} = w^{(k-1)} - \eta \, \nabla_w L(w^{(k-1)}, b^{(k-1)})$, is the **gradient step on the coefficients** from Half 1 of the proximal update. It says: at step $k$, the intermediate vector $z^{(k)}$ is obtained by taking a gradient step from the previous coefficient vector $w^{(k-1)}$, using the gradient of $L$ evaluated at the previous $(w^{(k-1)}, b^{(k-1)})$.
- The second line, $w^{(k)} = S_{\eta \alpha}(z^{(k)})$, is the **soft-thresholding step** from Half 2. It says: at step $k$, the new coefficient vector $w^{(k)}$ is the soft-thresholded version of the intermediate $z^{(k)}$ produced by the previous line.
- The third line, $b^{(k)} = b^{(k-1)} - \eta \, \nabla_b L(w^{(k-1)}, b^{(k-1)})$, is the **plain gradient step on the intercept**. It says: at step $k$, the new intercept $b^{(k)}$ is obtained by taking a gradient step from the previous intercept $b^{(k-1)}$.

So the invariant is the entire story of training compressed into a recurrence relation: each $w^{(k)}$ and $b^{(k)}$ in the sequence must follow exactly from the previous one via the right update rules, all the way back to the starting point $w^{(0)} = 0$ and $b^{(0)} = 0$. If `self.coef_` and `self.intercept_` always satisfy this recurrence, we know the algorithm is doing the right computation at every step.

The proof has two parts, corresponding to the two domino conditions from the introduction:

1. **Base case (the first domino falls):** Show the invariant holds before the very first iteration.
2. **Inductive step (each domino knocks over the next):** Show that if the invariant holds at iteration $t$, it still holds at iteration $t + 1$.

If both parts go through, induction gives us the invariant for every iteration, which means the algorithm's output is correct for every training run.

----------

## Base case: before the first iteration ($t = 0$)

**What we need to show:** the invariant holds at the very start, before any training step has been taken.

Right before the loop starts, the code initializes both stored values to zero:

```python
self.coef_      = np.zeros(n_features)   # w^(0) = 0
self.intercept_ = 0.0                    # b^(0) = 0
```

For $t = 0$, the invariant's list "for every $k = 1, 2, \ldots, 0$" is an **empty list**. There are no values of $k$ to check, because no steps have been taken yet. The invariant therefore makes no factual claims about $t = 0$, and any factual claim about an empty list is automatically true (this is sometimes called a "vacuously true" statement). The base case holds.

----------

## Inductive step: from iteration $t$ to iteration $t+1$

**What we assume (the induction hypothesis):** the invariant already holds at the start of iteration $t$. So `self.coef_` $= w^{(t)}$ and `self.intercept_` $= b^{(t)}$, where $w^{(t)}$ and $b^{(t)}$ are the values produced by $t$ correct steps.

**What we need to show:** after the body of the loop runs one more time, the invariant still holds at iteration $t+1$. That is, the new values stored in `self.coef_` and `self.intercept_` are exactly the $w^{(t+1)}$ and $b^{(t+1)}$ demanded by the recurrence.

The loop body has three logical sub-steps, corresponding to the three lines of the recurrence. We verify each one.

### Sub-step (a): gradient step on the coefficients

The relevant code:

```python
p_hat              = self._sigmoid(X @ self.coef_ + self.intercept_)
residual           = p_hat - y
weighted_residual  = sw * residual
grad_coef          = (X.T @ weighted_residual) / sw_sum
coef_new           = self.coef_ - self.learning_rate * grad_coef
```

The intermediate `residual` is the prediction error on each training row: how far the model's probability $\hat{p}_i$ landed from the true label $y_i$. If the model said 0.8 but the comment is actually non-toxic ($y_i = 0$), the residual is $+0.8$, meaning "the model leaned toxic when it should have leaned clean." Multiplying by `sw` weighs each row's contribution, so rare-class samples (boosted via `class_weight="balanced"`) contribute more to the gradient.

The gradient of the weighted logistic loss with respect to $w$ is

$$\nabla_w L(w^{(t)}, b^{(t)}) = \frac{X^\top \bigl(\mathbf{sw} \odot (\hat{p}^{(t)} - y)\bigr)}{\sum_i \mathbf{sw}_i}$$

There is some new notation here, so let us spell it out. The symbol $\odot$ is **element-wise multiplication** (also called the Hadamard product): you multiply two vectors of the same length entry by entry to get another vector of the same length. So $\mathbf{sw} \odot (\hat{p}^{(t)} - y)$ is a vector whose $i$-th entry is $\mathbf{sw}_i \cdot (\hat{p}_i^{(t)} - y_i)$: each residual gets scaled by its sample weight. The $X^\top \cdot (\text{this vector})$ part is a matrix-vector multiplication, which produces a new vector with $p$ entries (one per feature). The division by $\sum_i \mathbf{sw}_i$ normalizes by the total weight, so step sizes are comparable whether you have many heavily-weighted rows or many lightly-weighted ones.

When all sample weights equal 1, the formula reduces to the standard unweighted form $\frac{X^\top (\hat{p}^{(t)} - y)}{n}$, which you may recognize from a textbook derivation of logistic regression gradients.

The code computes exactly this gradient line by line: it forms $\hat{p}$, subtracts $y$, multiplies by $\mathbf{sw}$, hits the result with $X^\top$, and divides by $\sum \mathbf{sw}_i$. Then it takes a step in the opposite direction:

$$\texttt{coef\_new} = w^{(t)} - \eta \, \nabla_w L(w^{(t)}, b^{(t)})$$

Why the opposite direction? The gradient points uphill (the direction of steepest *increase* in loss). To minimize the loss, we want to go downhill, so we subtract the gradient instead of adding it.

The right-hand side here is exactly the definition of $z^{(t+1)}$ from the first line of the recurrence. So the code's `coef_new` after this block equals $z^{(t+1)}$. **This sub-step matches the first line of the recurrence exactly.**

### Sub-step (b): soft-thresholding (the proximal step)

The next line of code applies the proximal operator that handles the L1 penalty:

```python
coef_new = self._soft_threshold(coef_new, self.alpha * self.learning_rate)
```

The soft-thresholding operator is defined as

$$S_\lambda(z) = \operatorname{sign}(z) \cdot \max(|z| - \lambda, 0)$$

The notation $\operatorname{sign}(z)$ returns $+1$ if $z$ is positive, $-1$ if $z$ is negative, and $0$ if $z$ is zero. When $z$ is a vector, $\operatorname{sign}(z)$ is applied entry by entry, and so is the rest of the formula.

In plain words: look at each coefficient individually. If its magnitude is smaller than $\lambda$, set it to zero (the feature gets removed from the model). If its magnitude is larger, shrink it toward zero by exactly $\lambda$ but keep its original sign. The result is sparser than the input.

For example, if $z = (3, -0.05, 0.1, -2)$ and $\lambda = 0.2$:
- $z_1 = 3$: magnitude is 3, which is bigger than 0.2, so we shrink it by 0.2 and keep the sign $\to +2.8$
- $z_2 = -0.05$: magnitude is 0.05, smaller than 0.2, so it becomes $0$
- $z_3 = 0.1$: magnitude is 0.1, smaller than 0.2, so it becomes $0$
- $z_4 = -2$: magnitude is 2, bigger than 0.2, so we shrink it by 0.2 and keep the sign $\to -1.8$

Result: $S_{0.2}(z) = (2.8, 0, 0, -1.8)$. Two entries got killed entirely; the other two got shrunk a little. That is L1 sparsity in action.

This operator is not an arbitrary fudge. It is the exact closed-form solution to the optimization problem

$$S_\lambda(z) = \arg\min_w \, \frac{1}{2} \|w - z\|^2 + \lambda \|w\|_1$$

The notation $\arg\min_w$ reads as "the value of $w$ that makes the expression smallest". So this says: $S_\lambda(z)$ is the vector $w$ that minimizes the expression $\frac{1}{2}\|w - z\|^2 + \lambda \|w\|_1$. The first part $\|w - z\|^2$ wants $w$ to be close to $z$; the second part $\lambda \|w\|_1$ wants $w$ to be sparse. The trade-off between these two is what the proximal operator captures, and soft-thresholding solves it exactly.

Plugging this fact into proximal gradient theory tells us that the two-step update (gradient step, then soft-threshold) is mathematically equivalent to performing one true gradient step on the full L1-penalized objective $J$.

The code uses $\lambda = \eta \alpha$, which is the standard choice for proximal gradient. By sub-step (a), the input to soft-thresholding is $z^{(t+1)}$. So after this line:

$$\texttt{coef\_new} = S_{\eta \alpha}(z^{(t+1)})$$

The right-hand side is exactly the definition of $w^{(t+1)}$ from the second line of the recurrence. So the code's `coef_new` after this block equals $w^{(t+1)}$. **This sub-step matches the second line of the recurrence exactly.**

### Sub-step (c): intercept update

The intercept gets a plain gradient step, no soft-thresholding:

```python
intercept_new = self.intercept_ - self.learning_rate * (weighted_residual.sum() / sw_sum)
```

The intercept does not appear inside the L1 penalty $\alpha \|w\|_1$ (only the coefficient vector $w$ does), so we never had a non-smooth corner to deal with for $b$ in the first place. Plain gradient descent works fine on this term.

The gradient of $L$ with respect to $b$ is

$$\nabla_b L(w^{(t)}, b^{(t)}) = \frac{\sum_i \mathbf{sw}_i \cdot (\hat{p}_i^{(t)} - y_i)}{\sum_i \mathbf{sw}_i}$$

which is just the weighted average of the residuals. The numerator $\sum_i \mathbf{sw}_i \cdot (\hat{p}_i^{(t)} - y_i)$ sums up "weight times residual" across all training rows, and dividing by $\sum_i \mathbf{sw}_i$ turns the sum into an average. The code computes this as `weighted_residual.sum() / sw_sum` and steps:

$$\texttt{intercept\_new} = b^{(t)} - \eta \, \nabla_b L(w^{(t)}, b^{(t)})$$

The right-hand side is exactly the definition of $b^{(t+1)}$ from the third line of the recurrence. So the code's `intercept_new` equals $b^{(t+1)}$. **This sub-step matches the third line of the recurrence exactly.**

(Side note: when `fit_intercept=False`, the intercept stays at zero and is never updated, so sub-step (c) is a no-op. The proof still holds since $b^{(k)} = 0$ for all $k$ in that case.)

### Step-size assumption (when does the loss actually decrease?)

For proximal gradient to make progress (rather than oscillate or overshoot), the learning rate has to be small enough. The standard condition is

$$\eta \leq \frac{1}{L_{\text{lip}}}$$

where $L_{\text{lip}}$ is the **Lipschitz constant** of the gradient $\nabla_w L$. A function's gradient is said to be **$L_{\text{lip}}$-Lipschitz** if, whenever you move the input by some amount, the gradient changes by at most $L_{\text{lip}}$ times that amount. Intuitively: the gradient does not change too suddenly. The constant $L_{\text{lip}}$ is a measure of how "wiggly" the function is. Smooth, gently-curved functions have small $L_{\text{lip}}$; functions with sharp turns have large $L_{\text{lip}}$.

The condition $\eta \leq 1/L_{\text{lip}}$ says the learning rate must be inversely proportional to how wiggly the loss is. A wigglier loss needs smaller steps because the gradient changes more abruptly, and a too-large step would overshoot. A smoother loss can tolerate bigger steps.

Under this assumption, the convexity of $L$ plus the Lipschitz property of $\nabla_w L$ give the standard proximal gradient descent guarantee:

$$J(w^{(t+1)}, b^{(t+1)}) \leq J(w^{(t)}, b^{(t)})$$

In words: every iteration makes the cost go down (or at least not up). The algorithm cannot get worse, only better or stationary.

One caveat: the code does not check or enforce this condition. The user picks `learning_rate` when constructing the model, and choosing too aggressive a value can cause the iterates to oscillate or diverge instead of converging. The convergence warning (Case 2 below) is the safety net for when this happens.

### Wrapping up the inductive step

Sub-steps (a), (b), and (c) together established that:
- `coef_new` (after both gradient and soft-threshold) $= w^{(t+1)}$
- `intercept_new` $= b^{(t+1)}$

The code then commits these into the stored attributes:

```python
self.coef_      = coef_new       # = w^(t+1)
self.intercept_ = intercept_new  # = b^(t+1)
```

At the start of the next iteration, `self.coef_` and `self.intercept_` contain $w^{(t+1)}$ and $b^{(t+1)}$. Together with the assumption (which we inherited as the induction hypothesis) that all earlier values $w^{(1)}, \ldots, w^{(t)}$ and $b^{(1)}, \ldots, b^{(t)}$ already obey the recurrence, the new $(w^{(t+1)}, b^{(t+1)})$ extends the recurrence by one step. The invariant now holds at $t + 1$. The induction goes through.

----------

## Termination

The loop exits in one of two ways.

### Case 1: Convergence

```python
if delta < self.tol:
    break
```

The variable $\delta$ measures the largest change to any coefficient or to the intercept during the last iteration:

$$\delta = \max\bigl(\|w^{(t+1)} - w^{(t)}\|_\infty, \; |b^{(t+1)} - b^{(t)}|\bigr)$$

The notation $\|v\|_\infty$ is called the **infinity norm** (or **max norm**) of a vector $v$. It just means the largest absolute value among its entries: $\|v\|_\infty = \max_j |v_j|$. For example, if $v = (3, -7, 2, 0)$, then $\|v\|_\infty = 7$ (the magnitude of the entry with the largest absolute value).

So $\|w^{(t+1)} - w^{(t)}\|_\infty$ measures "the biggest change in any single coefficient during this iteration." Taking the outer $\max$ with $|b^{(t+1)} - b^{(t)}|$ extends the question to include the intercept, so $\delta$ is the biggest single change in any parameter (whether a coefficient or the intercept).

Reading it intuitively: $\delta$ asks "what is the single biggest move any parameter just made?" If even the most-changing parameter moved by less than `tol` (a small number, $10^{-4}$ by default), the algorithm has essentially stopped moving.

When this branch fires, the iterates have approximately stopped changing. Because $J$ is convex (a bowl shape with one bottom), "we have stopped moving" can only mean "we are at the bottom of the bowl, up to the resolution of `tol`". A convex function has no side-valleys to get stuck in: if the gradient is small enough that the iterates barely move, the only place you can be is near the global minimum. So when the loop exits via this condition, the saved `self.coef_` and `self.intercept_` represent a near-optimal solution to $\min_{w, b} J(w, b)$.

### Case 2: Maximum iterations

```python
for iteration in range(self.max_iter):
```

If the loop reaches `max_iter` without ever satisfying $\delta < $ `tol`, it exits anyway. The model still has the result of $T = $ `max_iter` correct steps (by the invariant), so the output is not garbage. But the algorithm did not fully converge, which usually means one of three things: `max_iter` was too small for the chosen `learning_rate`, the learning rate was too small to make meaningful progress in the budget, or the learning rate was too large and the iterates are oscillating without settling.

The code issues a `ConvergenceWarning` so the user knows to either increase `max_iter`, adjust the learning rate, or accept the partial result.

----------

## Conclusion

The base case shows the invariant holds before any iteration runs. The inductive step shows that one iteration of the loop body preserves it. By induction on $t$ (the dominoes fall, all of them), the invariant holds at every iteration. So `self.coef_` and `self.intercept_` always contain a correctly-computed sequence of proximal gradient iterates of the L1-penalized weighted logistic loss.

When the algorithm exits (either via convergence or max-iterations), these values represent an (approximately) optimal solution to

$$\min_{w,\, b} \; L(w, b) + \alpha \|w\|_1$$

under the conditions that $J$ is convex (which it is, since logistic loss and the L1 norm are both convex) and that the learning rate was chosen small enough to satisfy the step-size assumption.